# Material Management Notebook

This notebook manages liquid materials registered in the MADSci Resource Manager.

**Prerequisites:** `madsci_postgres` and `resource_manager` services must be running.

```bash
docker compose up -d --build madsci_postgres event_manager resource_manager
```

To use section 5 (apply calibration results from Data Manager), also start:

```bash
docker compose up -d --build madsci_mongodb data_manager
```

**Operations:**
1. Setup — connect to Resource Manager and Data Manager
2. List — show all registered materials
3. Add — register a new material
4. Update (Pattern A) — manually edit attributes
5. Update (Pattern B) — apply calibration results from Data Manager
6. Remove — delete a material

---
## 1. Setup

In [9]:
from madsci.client.data_client import DataClient
from madsci.client.resource_client import ResourceClient
from madsci.common.types.resource_types import Consumable

# Resource Manager URL (default port 8003)
RESOURCE_MANAGER_URL = "http://localhost:8003/"

# Data Manager URL (default port 8004) — used in section 5
DATA_MANAGER_URL = "http://localhost:8004/"

# Resource class label used to identify high viscosity liquid materials
MATERIAL_CLASS = "high_viscosity_liquid"

# Current schema version — increment when attribute structure changes
CURRENT_SCHEMA_VERSION = "0.0"

resource_client = ResourceClient(resource_server_url=RESOURCE_MANAGER_URL)
data_client = DataClient(data_server_url=DATA_MANAGER_URL)

print(f"Resource Manager    : {RESOURCE_MANAGER_URL}")
print(f"Data Manager        : {DATA_MANAGER_URL}")
print(f"Material class      : {MATERIAL_CLASS}")
print(f"Schema version      : {CURRENT_SCHEMA_VERSION}")

2026-05-05T19:07:46.823204Z [info     ] EventClient initialized        client_name=madsci.client.data_client event_server='Not configured' log_dir=.madsci\logs log_level=EventLogLevel.INFO madsci_version=0.7.0 platform=Windows-11-10.0.26200-SP0 python_version=3.13.2


Resource Manager    : http://localhost:8003/
Data Manager        : http://localhost:8004/
Material class      : high_viscosity_liquid
Schema version      : 0.0


---
## Schema Version History

Track all structural changes to `attributes` here.  
When the structure changes, increment `CURRENT_SCHEMA_VERSION` and add a migration function in the next cell.

| Version | Date | Changes |
|---------|------|---------|
| 0.0 | 2026-05-05 | Initial schema |

In [ ]:
# Schema v0.0 — attribute structure reference
#
# resource_name  : unique material name (used as key in actions)
# resource_class : "high_viscosity_liquid"
#
# attributes:
#   schema_version: "0.0"
#
#   product_info:
#     material_name    (str, required): raw material name
#     common_name      (str)          : common / trade name
#     manufacturer     (str)          : supplier name
#     cas_number       (str)          : CAS registry number
#     lot_number       (str)          : lot / batch number
#     price_jpy_per_kg (float)        : unit price [JPY/kg]
#     application      (str)          : e.g. "plasticizer", "pigment", "hardener", "base resin"
#     chemical_class   (str)          : e.g. "amine", "silicone", "hydrocarbon"
#
#   physical_properties:
#     nominal:                         # Manufacturer's catalog / nominal values
#       density_g_per_cm3 (float): density [g/cm³]
#       viscosity_mPas    (float): viscosity [mPa·s]
#     measured:                        # Values obtained from calibration or in-house measurement
#       density_g_per_cm3 (float): density [g/cm³]
#       viscosity_mPas    (float): viscosity [mPa·s]
#
#   dispensing_params:
#     "{pressure_MPa}":          e.g. "0.1MPa"
#       g_per_rev           (float): grams dispensed per revolution
#       max_stable_speed_rps(float): maximum stable dispensing speed [rps]
#       calibrated_at       (str)  : calibration datetime (ISO format)
#       source_datapoint_id (str)  : datapoint_id in Data Manager


# --- Migration functions ---
# Add new functions below when schema version is incremented.
#
# def migrate_v0_0_to_v1_0(material):
#     """Migration from v0.0 to v1.0: describe what changed."""
#     material.attributes["product_info"]["new_field"] = None
#     material.attributes["schema_version"] = "1.0"
#     return material

print("Schema reference loaded.")

---
## 2. List All Registered Materials

In [16]:
from requests.exceptions import HTTPError

# Query all resources with resource_class == MATERIAL_CLASS
try:
    materials = resource_client.query_resource(resource_class=MATERIAL_CLASS, multiple=True)
except HTTPError as e:
    if e.response is not None and e.response.status_code == 404:
        materials = []
    else:
        raise

if not materials:
    print("No materials registered yet.")
else:
    print(f"{len(materials)} material(s) registered:\n")
    for m in materials:
        attrs = m.attributes or {}
        print(f"  name       : {m.resource_name}")
        print(f"  id         : {m.resource_id}")
        print(f"  attributes : {attrs}")
        print()


2 material(s) registered:



  name       : test1
  id         : 01KQWSTP58SAFZXFHV149YAZGG
  attributes : {'schema_version': '0.0', 'product_info': {'material_name': 'test1', 'common_name': 'test1', 'manufacturer': 'SEKISUI', 'cas_number': '11111111', 'lot_number': 'ABC123', 'price_jpy_per_kg': '1500', 'application': 'hardener', 'chemical_class': 'amine'}, 'physical_properties': {'nominal': {'density_g_per_cm3': 1.54, 'viscosity_mPas': 30}, 'measured': {'density_g_per_cm3': 1.22, 'viscosity_mPas': 44}}, 'dispensing_params': {}}

  name       : test2
  id         : 01KQWT16X6CRPHHCK2ATH8ZQ4C
  attributes : {'schema_version': '0.0', 'product_info': {'material_name': 'test2', 'common_name': 'test2', 'manufacturer': 'SEKISUI', 'cas_number': '222222', 'lot_number': '1234', 'price_jpy_per_kg': '150', 'application': 'plasticizer'}, 'physical_properties': {'nominal': {'density_g_per_cm3': 1.54}, 'measured': {'density_g_per_cm3': 1.22}}, 'dispensing_params': {}}



---
## 3. Add a New Material

Edit `MATERIAL_NAME` and `ATTRIBUTES` before running.

In [15]:
# --- Edit here ---
MATERIAL_NAME = "test2"   # Unique name used as key in actions

ATTRIBUTES = {
    "schema_version": CURRENT_SCHEMA_VERSION,
    "product_info": {
        "material_name":    MATERIAL_NAME,   # Raw material name (required)
        "common_name":      "test2",            # Common / trade name
        "manufacturer":     "SEKISUI",            # Supplier name
        "cas_number":       "222222",            # CAS registry number
        "lot_number":       "1234",            # Lot / batch number
        "price_jpy_per_kg": "150",            # Unit price [JPY/kg]
        "application":      "plasticizer",            # e.g. "plasticizer", "pigment", "hardener"
        #"chemical_class":   "amine",            # e.g. "amine", "silicone", "hydrocarbon"
    },
    "physical_properties": {
        "nominal": {                         # Manufacturer's catalog / nominal values
            "density_g_per_cm3": 1.54,       # Density [g/cm³]
            #"viscosity_mPas":    30,       # Viscosity [mPa·s]
        },
        "measured": {                        # Values from calibration or in-house measurement
            "density_g_per_cm3": 1.22,       # Density [g/cm³]
            #"viscosity_mPas":    44,       # Viscosity [mPa·s]
        },
    },
    "dispensing_params": {
        # Populated by calibration — see section 5 (Pattern B)
        # "0.1MPa": {
        #     "g_per_rev": None,
        #     "max_stable_speed_rps": None,
        #     "calibrated_at": None,
        #     "source_datapoint_id": None,
        # }
    },
}
# -----------------

new_material = Consumable(
    resource_name=MATERIAL_NAME,
    resource_class=MATERIAL_CLASS,
    attributes=ATTRIBUTES,
)
result = resource_client.add_resource(new_material)
print(f"Registered: {result.resource_name} (id: {result.resource_id})")
print(f"schema_version: {result.attributes.get('schema_version')}")

Registered: test2 (id: 01KQWT16X6CRPHHCK2ATH8ZQ4C)
schema_version: 0.0


---
## 4. Update (Pattern A) — Manual

Directly specify the new name or attributes to update.  
Use this when you want to edit basic material information.

In [14]:
# --- Edit here ---
TARGET_NAME = "test1"   # Current name of the material to update
NEW_NAME    = "test1"   # New name (leave unchanged if not renaming)

# Only fill in the fields you want to change — others will be preserved.
PRODUCT_INFO_UPDATES = {
    # "material_name":    "Material_A",
    # "common_name":      "Mat-A",
    # "manufacturer":     "Sigma-Aldrich",
    # "cas_number":       "123-45-6",
     "lot_number":       "ABC123",
    # "price_jpy_per_kg": 5000,
    # "application":      "plasticizer",
    # "chemical_class":   "silicone",
}
NOMINAL_PROPERTIES_UPDATES = {
    # Manufacturer's catalog / nominal values
    # "density_g_per_cm3": 1.05,
    # "viscosity_mPas":    50000,
}
MEASURED_PROPERTIES_UPDATES = {
    # Values from calibration or in-house measurement
    # "density_g_per_cm3": 1.03,
    # "viscosity_mPas":    48000,
}
# -----------------

# Fetch the existing resource
target = resource_client.query_resource(
    resource_name=TARGET_NAME,
    resource_class=MATERIAL_CLASS,
    unique=True,
)

# Apply changes (merge into existing attributes)
attrs = target.attributes or {}
if PRODUCT_INFO_UPDATES:
    attrs["product_info"] = {**attrs.get("product_info", {}), **PRODUCT_INFO_UPDATES}
if NOMINAL_PROPERTIES_UPDATES:
    physical = attrs.get("physical_properties", {})
    physical["nominal"] = {**physical.get("nominal", {}), **NOMINAL_PROPERTIES_UPDATES}
    attrs["physical_properties"] = physical
if MEASURED_PROPERTIES_UPDATES:
    physical = attrs.get("physical_properties", {})
    physical["measured"] = {**physical.get("measured", {}), **MEASURED_PROPERTIES_UPDATES}
    attrs["physical_properties"] = physical
target.resource_name = NEW_NAME
target.attributes = attrs

updated = resource_client.update_resource(target)
print(f"Updated: {updated.resource_name} (id: {updated.resource_id})")
print(f"attributes: {updated.attributes}")

Updated: test1 (id: 01KQWSTP58SAFZXFHV149YAZGG)
attributes: {'schema_version': '0.0', 'product_info': {'material_name': 'test1', 'common_name': 'test1', 'manufacturer': 'SEKISUI', 'cas_number': '11111111', 'lot_number': 'ABC123', 'price_jpy_per_kg': '1500', 'application': 'hardener', 'chemical_class': 'amine'}, 'physical_properties': {'nominal': {'density_g_per_cm3': 1.54, 'viscosity_mPas': 30}, 'measured': {'density_g_per_cm3': 1.22, 'viscosity_mPas': 44}}, 'dispensing_params': {}}


---
## 5. Update (Pattern B) — Apply Calibration Results from Data Manager

Fetch past calibration results for a material from the Data Manager,
select the result you want to apply, and write it back to the Resource Manager.

**Prerequisites:** `madsci_mongodb` and `data_manager` must be running.

In [ ]:
# --- Edit here ---
TARGET_NAME = "Material_A"   # Material name to fetch calibration results for
# -----------------

# Step 1: Fetch all calibration results for this material from Data Manager
all_datapoints = data_client.query_datapoints({
    "label": "calibration_results",
    "value.metadata.material_name": TARGET_NAME,
})

if not all_datapoints:
    print(f"No calibration results found for '{TARGET_NAME}' in Data Manager.")
else:
    # Sort by timestamp (newest first) and display
    sorted_points = sorted(
        all_datapoints.values(),
        key=lambda d: d.data_timestamp,
        reverse=True,
    )
    print(f"Found {len(sorted_points)} calibration result(s) for '{TARGET_NAME}':\n")
    for i, dp in enumerate(sorted_points):
        meta = dp.value.get("metadata", {})
        results = dp.value.get("results", [])
        best = max(results, key=lambda r: r["g_per_rev"]) if results else {}
        print(f"  [{i}] {dp.data_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"       best g/rev       : {best.get('g_per_rev', 'N/A')}")
        print(f"       best speed [rps] : {best.get('speed_rps', 'N/A')}")
        print(f"       datapoint_id     : {dp.datapoint_id}")
        print()

In [ ]:
# --- Edit here ---
SELECTED_INDEX = 0       # Index of the calibration result to apply (from the list above)
PRESSURE_KEY   = "0.1MPa"  # Pressure condition key to store under dispensing_params
# -----------------

selected = sorted_points[SELECTED_INDEX]
results = selected.value.get("results", [])

# Pick the result with the highest g/rev as the representative value
best = max(results, key=lambda r: r["g_per_rev"])
g_per_rev            = best["g_per_rev"]
max_stable_speed_rps = best["speed_rps"]

print(f"Applying calibration result [{SELECTED_INDEX}] under key '{PRESSURE_KEY}':")
print(f"  g_per_rev            : {g_per_rev}")
print(f"  max_stable_speed_rps : {max_stable_speed_rps}")

# Fetch material from Resource Manager and update dispensing_params
material = resource_client.query_resource(
    resource_name=TARGET_NAME,
    resource_class=MATERIAL_CLASS,
    unique=True,
)
attrs = material.attributes or {}
dispensing_params = attrs.get("dispensing_params", {})
dispensing_params[PRESSURE_KEY] = {
    "g_per_rev":            g_per_rev,
    "max_stable_speed_rps": max_stable_speed_rps,
    "calibrated_at":        selected.data_timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
    "source_datapoint_id":  selected.datapoint_id,
}
attrs["dispensing_params"] = dispensing_params
material.attributes = attrs

updated = resource_client.update_resource(material)
print(f"\nResource Manager updated: {updated.resource_name}")
print(f"dispensing_params: {updated.attributes.get('dispensing_params')}")

---
## 6. Remove a Material

Set `TARGET_NAME` to the name of the material to remove.  
The resource is moved to history (soft delete) — it is not permanently deleted.

In [ ]:
# --- Edit here ---
TARGET_NAME = "Material_A"   # Name of the material to remove
# -----------------

# Fetch the resource first to get its ID
target = resource_client.query_resource(
    resource_name=TARGET_NAME,
    resource_class=MATERIAL_CLASS,
    unique=True,
)

removed = resource_client.remove_resource(target)
print(f"Removed: {removed.resource_name} (id: {removed.resource_id})")